# Statistics, Experimentation & Causal Inference

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/08_statistics.ipynb)

Runnable examples from **Data/ML Handbook** — section `08_statistics`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/08_statistics/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s06_mle_bh.py`

In [ ]:
import numpy as np
from scipy import stats, optimize
from statsmodels.stats.multitest import multipletests
x = np.random.default_rng(0).normal(3, 1.5, 500)
nll = lambda th: -stats.norm(th[0], np.exp(th[1])).logpdf(x).sum()     # optimise log σ (unconstrained)
res = optimize.minimize(nll, [0, 0])
print(f"numeric MLE μ={res.x[0]:.4f} σ={np.exp(res.x[1]):.4f} | closed form μ={x.mean():.4f} σ={x.std(ddof=0):.4f}")
rng = np.random.default_rng(1)
p = np.r_[rng.uniform(0, 1, 90), rng.uniform(0, 0.001, 10)]            # 90 nulls, 10 real effects
for m in ["bonferroni", "holm", "fdr_bh"]:
    rej = multipletests(p, alpha=0.05, method=m)[0]
    print(f"{m:10s}: discoveries={rej.sum():2d}  true={rej[90:].sum()}  false={rej[:90].sum()}")
print("uncorrected:", (p < .05).sum(), "discoveries, false =", (p[:90] < .05).sum())

### `code/s06_tests.py`

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(0)
a, b = rng.normal(100, 15, 40), rng.normal(108, 25, 45)
t = stats.ttest_ind(a, b, equal_var=False)                  # Welch by default in practice
d = (b.mean() - a.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
print(f"Welch t={t.statistic:.2f} p={t.pvalue:.4f}  Cohen's d={d:.2f}")
print("Mann-Whitney p =", round(stats.mannwhitneyu(a, b).pvalue, 4))
tab = pd.crosstab(rng.choice(["A", "B"], 500), rng.choice(["buy", "no"], 500, p=[.3, .7]))
chi2, p, dof, exp = stats.chi2_contingency(tab); print(f"chi2={chi2:.2f} dof={dof} p={p:.3f}")
g1, g2, g3 = rng.normal(5, 1, 30), rng.normal(5.2, 1, 30), rng.normal(6, 1, 30)
print("ANOVA F=%.2f p=%.5f" % stats.f_oneway(g1, g2, g3))
x = rng.lognormal(3, .8, 200); boot = [np.median(rng.choice(x, len(x))) for _ in range(5000)]
print("median %.2f, 95%% bootstrap CI [%.2f, %.2f]" % (np.median(x), *np.percentile(boot, [2.5, 97.5])))

### `code/s06_ab.py`

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
n = NormalIndPower().solve_power(proportion_effectsize(0.11, 0.10), alpha=0.05, power=0.8)
print(f"needed per arm for 10% -> 11%: {int(np.ceil(n)):,}  (rule of thumb 16·p(1−p)/δ² = {16*.1*.9/.01**2:,.0f})")
conv, nobs = np.array([1_580, 1_450]), np.array([14_500, 14_500])        # B, A
z, p = proportions_ztest(conv, nobs)
lo, hi = confint_proportions_2indep(conv[0], nobs[0], conv[1], nobs[1])
print(f"rates B={conv[0]/nobs[0]:.4f} A={conv[1]/nobs[1]:.4f}  z={z:.2f} p={p:.4f}  diff 95% CI [{lo:.4f}, {hi:.4f}]")
from scipy.stats import chisquare
print("SRM check p =", round(chisquare([14_500, 14_210]).pvalue, 4), "(< 0.001 would mean broken randomisation)")
rng = np.random.default_rng(0)                                            # CUPED variance reduction
pre = rng.normal(50, 10, 10_000); post = pre * 0.8 + rng.normal(0, 5, 10_000)
theta = np.cov(pre, post)[0, 1] / pre.var(); cuped = post - theta * (pre - pre.mean())
print(f"CUPED: variance {post.var():.1f} -> {cuped.var():.1f} ({1-cuped.var()/post.var():.0%} reduction)")